# 🧠 Python OOP + Django REST Framework Serializer Cheat Sheet

A practical learning guide based on the concepts discussed: **Python OOP → DRF serializers → inheritance → overriding → DRF hooks → custom methods → validation → nested serializers**.

> **Core idea:** A method is just a Python method. Its special behavior comes from the framework recognizing its name/interface and calling it at the right time.

## 1. The Big Picture

A DRF serializer is a **Python class**.

```python
class StudentSerializer(serializers.ModelSerializer):
    ...
```

Mental model:

```text
Python OOP
   ↓
Class
   ↓
Inheritance
   ↓
ModelSerializer
   ↓
Your Serializer
   ↓
DRF automatically uses certain methods
```

## 2. Class, Object, and Method

### Class
A class is a blueprint.

### Object / Instance
When you write:

```python
serializer = StudentSerializer(student)
```

`serializer` is an **instance/object** of `StudentSerializer`.

### Method
A function defined inside a class is called a **method**.

```python
class StudentSerializer(serializers.ModelSerializer):

    def hello(self):
        return "Hello"
```

`hello()` is a method.

## 3. What is `self`?

`self` refers to the **current object/instance**.

```python
class StudentSerializer(serializers.ModelSerializer):

    def hello(self):
        return "Hello"

    def test(self):
        return self.hello()
```

So:

```python
self.hello()
```

means:

> Call `hello()` belonging to this serializer object.

## 4. Inheritance ⭐

This:

```python
class StudentSerializer(serializers.ModelSerializer):
```

means `StudentSerializer` **inherits** from `ModelSerializer`.

Conceptually:

```text
StudentSerializer
      ↓ inherits from
ModelSerializer
      ↓ inherits from
Serializer
```

Your serializer gets functionality from its parent classes and can customize selected behavior.

## 5. Method Overriding ⭐

Suppose the parent class already provides:

```python
def create(self, validated_data):
    ...
```

Your subclass can provide its own version:

```python
class StudentSerializer(serializers.ModelSerializer):

    def create(self, validated_data):
        # Your custom creation logic
        ...
```

This is called **method overriding**.

The same idea applies to:

- `create()`
- `update()`
- `validate()`
- `to_representation()`

## 6. DRF Hooks ⭐⭐⭐

Some method names have special meaning because DRF knows those names and calls them at particular stages.

| Method | Purpose |
|---|---|
| `create()` | Create a model instance when saving new data |
| `update()` | Update an existing model instance |
| `validate()` | Object-level / multi-field validation |
| `to_representation()` | Customize object → serialized output |
| `validate_<fieldname>()` | Validate one particular serializer field |

### Important

These names are not random. If you want DRF to automatically use a particular hook, you need to follow the method name/interface DRF expects.

## 7. Can You Create Your Own Methods?

**Yes!**

You can create any method you want:

```python
class StudentSerializer(serializers.ModelSerializer):

    def calculate_status(self, age):
        if age >= 18:
            return "Adult"
        return "Minor"
```

`calculate_status()` is **your own method**.

DRF does not automatically call it.

You call it yourself:

```python
serializer.calculate_status(20)
```

Or from another method:

```python
self.calculate_status(age)
```

### Easy distinction

```text
create()                → DRF knows it
update()                → DRF knows it
validate()              → DRF knows it
to_representation()    → DRF knows it

calculate_status()      → YOU created it
hello()                 → YOU created it
my_method()             → YOU created it
```

## 8. `validate_<fieldname>()` ⭐

DRF uses this naming pattern for field-level validation:

```text
validate_<field_name>
```

Examples:

```python
def validate_age(self, value):
    ...

def validate_name(self, value):
    ...

def validate_email(self, value):
    ...
```

Mapping:

```text
age
 ↓
validate_age()

name
 ↓
validate_name()

email
 ↓
validate_email()
```

The naming pattern matters because DRF recognizes it.

## 9. Field Validation

Example:

```python
def validate_age(self, value):
    if value < 18:
        raise serializers.ValidationError(
            "Student age must be 18 or above."
        )

    return value
```

Flow:

```text
age value
   ↓
validate_age()
   ↓
Is age valid?
   ↓
YES → return value
NO  → ValidationError
```

## 10. Object-Level Validation

Use `validate()` when multiple fields need to be checked together.

```python
def validate(self, attrs):
    if attrs['course'] == 'BCA' and attrs['age'] < 18:
        raise serializers.ValidationError(
            "BCA student must be 18 or above."
        )

    return attrs
```

Easy distinction:

```text
validate_age()
    ↓
ONE field

validate_name()
    ↓
ONE field

validate()
    ↓
MULTIPLE fields
```

## 11. Your Student Validation Example

```python
class StudentSerializer(serializers.ModelSerializer):

    class Meta:
        model = Student
        fields = "__all__"

    def validate_age(self, value):
        if value < 18:
            raise serializers.ValidationError(
                "Student age must be 18 or above."
            )
        return value

    def validate_name(self, value):
        if not value.isalpha():
            raise serializers.ValidationError(
                "Student Name must contain alphabets."
            )
        return value

    def validate(self, attrs):
        if attrs['course'] == 'BCA' and attrs['age'] < 18:
            raise serializers.ValidationError(
                "BCA student must be 18 or above."
            )
        return attrs
```

### Note about `validate_age()`

If `age` is a Django `IntegerField`, DRF normally handles type conversion/validation before `validate_age()` is called. So the value will normally already be an integer.

Also, if you manually check types, check the type before doing a comparison such as `value < 18`.

## 12. `create()`

When valid serializer data is saved for a new object, DRF can use `create()`.

```python
def create(self, validated_data):
    student = Student.objects.create(**validated_data)
    return student
```

Typical flow:

```text
POST
 ↓
Serializer validation
 ↓
serializer.save()
 ↓
create()
 ↓
Database
```

### Memory rule

**CREATE → `create()`**

## 13. `update()`

When an existing object is updated, DRF can use `update()`.

```python
def update(self, instance, validated_data):
    instance.name = validated_data.get(
        'name',
        instance.name
    )

    instance.save()

    return instance
```

Typical idea:

```text
PUT / PATCH
     ↓
serializer.save()
     ↓
update()
     ↓
Database updated
```

### Memory rule

**UPDATE → `update()`**

## 14. `to_representation()`

This method is about **output**.

Conceptually:

```text
Database object
      ↓
to_representation()
      ↓
Python dictionary
      ↓
JSON response
```

Example:

```python
def to_representation(self, instance):
    data = super().to_representation(instance)

    data['message'] = 'Hello'

    return data
```

You can use it to customize what the API sends back.

## 15. `super()` ⭐

`super()` lets you access behavior from the parent class.

```python
data = super().to_representation(instance)
```

This means:

> Run the parent's `to_representation()` implementation.

Then you can customize the result:

```python
data = super().to_representation(instance)
data['extra'] = 'something'
return data
```

Mental model:

```text
Parent implementation
       ↓
     super()
       ↓
Your customization
```

## 16. Serializer `context`

DRF can provide extra information to a serializer through:

```python
self.context
```

A request is commonly available when the serializer is created by a DRF view.

Example:

```python
def to_representation(self, instance):
    data = super().to_representation(instance)

    req = self.context.get('request')

    print(req.user)
    print(req.method)

    return data
```

Mental model:

```text
Serializer
   ↓
self.context
   ↓
request
   ├── user
   └── method
```

## 17. `.get()` vs `[]`

This is normal Python dictionary behavior.

### `.get()`

```python
request = self.context.get('request')
```

If the key is missing:

```text
returns None
```

### `[]`

```python
request = self.context['request']
```

If the key is missing:

```text
raises KeyError
```

So:

```text
.get('request')
     ↓
safe if missing

['request']
     ↓
KeyError if missing
```

If you see a `KeyError`, it means the dictionary does not contain that key at that moment.

## 18. Why `self.context['request']` Might Raise `KeyError`

`self.context['request']` only works when the serializer's context actually contains a `request` key.

For example, if you manually instantiate a serializer without passing context, the key may not exist.

Conceptually:

```python
serializer = StudentSerializer(student)
```

may have no request in its context.

Whereas a DRF view commonly supplies context when it creates the serializer.

So:

```python
self.context.get('request')
```

is safer when the request may not exist.

## 19. Writable Nested Serializer

Your example:

```python
class EmployeeWritableNestedSerializer(serializers.ModelSerializer):

    department = DepartmentSerializer()

    class Meta:
        model = Employee
        fields = "__all__"
```

Conceptually:

```text
Employee
 ├── name
 ├── salary
 └── department
       ├── name
       └── ...
```

Nested serializers can be used for nested data. When you want to **write** nested data, custom `create()` / `update()` logic may be required.

## 20. Nested `create()` — Your Example

```python
def create(self, validated_data):
    departmentdata = validated_data.pop('department')

    department = Department.objects.create(
        **departmentdata
    )

    employee = Employee.objects.create(
        department=department,
        **validated_data
    )

    return employee
```

Flow:

```text
validated_data
      ↓
pop department
      ↓
department data separated
      ↓
Create Department
      ↓
Create Employee
      ↓
Connect Employee → Department
```

## 21. Nested `update()` — Your Example

```python
def update(self, instance, validated_data):
    departmentdata = validated_data.pop(
        'department',
        None
    )

    instance.name = validated_data.get(
        'name',
        instance.name
    )

    instance.salary = validated_data.get(
        'salary',
        instance.salary
    )

    if departmentdata:
        instance.department.name = departmentdata.get(
            'name',
            instance.department.name
        )
        instance.department.save()

    instance.save()

    return instance
```

This custom `update()` lets you update both:

```text
Employee
   +
Department
```

instead of only updating the main Employee fields.

## 22. OOP Concepts in Your DRF Code

| Your DRF code | OOP concept |
|---|---|
| `class StudentSerializer` | Class |
| `StudentSerializer(...)` | Object / instance |
| `self` | Current object |
| `ModelSerializer` | Parent/base class |
| `class X(ModelSerializer)` | Inheritance |
| Custom `create()` | Method overriding |
| Custom `update()` | Method overriding |
| `super()` | Access parent implementation |
| `calculate_status()` | Your own method |
| Same method name, different implementation | Polymorphism |

## 23. Polymorphism

Imagine:

```python
class StudentSerializer(serializers.ModelSerializer):

    def create(self, validated_data):
        # Student creation logic
        ...


class EmployeeSerializer(serializers.ModelSerializer):

    def create(self, validated_data):
        # Employee creation logic
        ...
```

Both classes have:

```python
create()
```

but they can behave differently.

Basic idea:

```text
Same interface/name
       ↓
Different behavior
```

That is the basic idea of **polymorphism**.

## 24. Complete Mental Map 🗺️

```text
                         Python
                           │
                         OOP
                           │
              ┌────────────┴────────────┐
              │                         │
            Class                    Object
              │                         │
     StudentSerializer          serializer = ...
              │
        Inheritance
              │
              ▼
       ModelSerializer
              │
       ┌──────┴─────────┐
       │                │
   Inherited        Your methods
   methods               │
       │          ┌──────┴──────────────┐
       │          │                     │
       ▼          ▼                     ▼
   create()    validate_age()      calculate_status()
   update()    validate_name()     hello()
   validate()  validate()          ...
   to_representation()
       │
       ▼
  DRF recognizes
  certain names
       │
       ▼
  Automatically
  calls them
```

## 25. Most Important Mental Model ⭐⭐⭐

Don't just memorize:

> "`create()` is for POST."

Instead understand the framework mechanism:

```text
DRF has a serializer system
        ↓
ModelSerializer provides methods
        ↓
You inherit those methods
        ↓
You can override selected methods
        ↓
DRF recognizes certain method names
        ↓
DRF calls them at the appropriate time
```

And separately:

```text
You create a method
        ↓
DRF does not automatically know it
        ↓
You call it yourself
```

## 26. One Rule to Remember ⭐⭐⭐

> **A method is just a Python method. Its special behavior comes from the framework recognizing its name/interface and calling it at the right time.**

Examples:

```python
def calculate_salary(self):
    # Your own method
    pass

def create(self, validated_data):
    # DRF-recognized method
    pass

def validate_age(self, value):
    # DRF field-validation hook
    # because of validate_<fieldname> naming pattern
    pass
```

## 27. Quick Revision Table

| Concept | Remember |
|---|---|
| Class | Blueprint |
| Object | Instance of a class |
| Method | Function inside a class |
| `self` | Current object |
| Inheritance | Child gets behavior from parent |
| Overriding | Child provides its own version of a parent method |
| `super()` | Access parent implementation |
| DRF hook | Method DRF knows and calls |
| Custom method | Method you create and call yourself |
| `validate_<field>` | Single-field validation |
| `validate()` | Multiple-field/object validation |
| `create()` | New object |
| `update()` | Existing object |
| `to_representation()` | Customize output |
| `context` | Extra information supplied to serializer |
| `.get()` | Missing dictionary key → `None` |
| `[]` | Missing dictionary key → `KeyError` |
| Writable nested serializer | Custom create/update may be needed |

## 🎯 Final Memory Formula

```text
Class
  +
Inheritance
  +
Methods
  +
Method Overriding
  +
self
  +
super()
  +
Framework Hooks
  +
Custom Methods
  +
Validation
  +
Nested Serializers
  =
Understanding DRF Serializers
```

### The key question to ask whenever you see a method

> **Is this a normal Python method that I created, or is this a method name that DRF recognizes and automatically calls?**

If you can answer that, you will understand a large part of how DRF serializers work.